#### Check file system

In [0]:
display(dbutils.fs.ls("abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/bronze"))

### Reading Raw Data

In [0]:
# LER OS DADOS
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

bronze_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/bronze/"

df_bronze = (
    spark.read
    .option("multiline", "true")
    .option("mode", "PERMISSIVE")
    .json(bronze_path)
    .withColumn("source_file", F.input_file_name())
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

display(df_bronze)
df_bronze.printSchema()

##### Utility Function

In [0]:
def clean_text(col_name):
    return F.trim(F.regexp_replace(F.col(col_name), r"\s+", " "))

#### Normalize Text Columns

In [0]:
# 1) Padronizar colunas textuais
text_cols = ["title", "company", "location", "description", "salary"]

df = df_bronze

for c in text_cols:
    if c in df.columns:
        df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))

for c in text_cols:
    if c in df.columns:
        df = df.withColumn(c, F.lower(F.col(c)))

#### Normalize Array String

In [0]:
# 2) Normalizar Skill separadamente por se tratar de um arry de strings
df = df.withColumn(
    "skills",
    F.expr("""
        filter(
            transform(
                skills,
                x -> trim(lower(regexp_replace(x, '\\\\s+', ' ')))
            ),
            x -> x is not null and x <> ''
        )
    """)
)

#### Sanitize NULL Values

In [0]:
# 3) Tratar nulos e strings vazias
for c in ["title", "company", "location", "description", "salary"]:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.when(F.col(c).isin("", "null", "none", "n/a", "na", "-"), None)
             .otherwise(F.col(c))
        )

# 3) Tratar nulos e strings vazias da coluna Skill separadamente
df = df.withColumn(
    "skills",
    F.expr("""
        filter(
            transform(
                skills,
                x -> case
                    when x in ('', 'null', 'none', 'n/a', 'na', '-') then null
                    else x
                end
            ),
            x -> x is not null
        )
    """)
)

#### Normalize Locations

In [0]:
# 5) Normalizar localização
df = (
    df
    .withColumn(
        "location_normalized",
        F.when(F.col("location").rlike(r"são paulo|sao paulo|sp, brasil|são paulo, sp|sao paulo, sp"), "sao paulo, sp")
         .when(F.col("location").rlike(r"rio de janeiro|rj, brasil|rio, rj"), "rio de janeiro, rj")
         .when(F.col("location").rlike(r"belo horizonte|bh, mg"), "belo horizonte, mg")
         .otherwise(F.col("location"))
    )
 )

#### Normalize Salary Data

In [0]:

# 6) Normalizar salário
df = df.withColumn("salary_raw", F.col("salary"))

# Extrair moeda e período
df = (
    df
    .withColumn(
        "salary_currency",
        F.when(F.col("salary").rlike(r"r\$|brl"), "BRL")
         .when(F.col("salary").rlike(r"usd|\$"), "USD")
         .otherwise(None)
    )
    .withColumn(
        "salary_period",
        F.when(F.col("salary").rlike(r"mês|mes|monthly|month"), "monthly")
         .when(F.col("salary").rlike(r"ano|anual|year|yearly"), "yearly")
         .when(F.col("salary").rlike(r"hora|hour|hourly"), "hourly")
         .otherwise("monthly")
    )
)

# Limpar texto do salário
df = (
    df
    .withColumn("salary_clean", F.regexp_replace(F.col("salary"), r"r\$|brl|usd|\$", ""))
    .withColumn("salary_clean", F.regexp_replace(F.col("salary_clean"), r"\s+", ""))
    .withColumn("salary_clean", F.regexp_replace(F.col("salary_clean"), r"(\d+)k", r"\1000"))
)

# Extrair mínimo e máximo
df = (
    df
    .withColumn("salary_min_str", F.regexp_extract(F.col("salary_clean"), r"(\d+[.,]?\d*)", 1))
    .withColumn("salary_max_str", F.regexp_extract(F.col("salary_clean"), r"[-a]\s*(\d+[.,]?\d*)", 1))
    .withColumn("salary_min_str", F.regexp_replace(F.col("salary_min_str"), ",", "."))
    .withColumn("salary_max_str", F.regexp_replace(F.col("salary_max_str"), ",", "."))
    .withColumn("salary_min", F.col("salary_min_str").cast("double"))
    .withColumn(
        "salary_max",
        F.when(F.col("salary_max_str") != "", F.col("salary_max_str").cast("double"))
         .otherwise(F.col("salary_min"))
    )
)

#### Normalize Skills

In [0]:
# 7) Limpar e normalizar skills
df = df.withColumn(
    "skills_array",
    F.expr("""
        filter(
            transform(
                skills,
                x -> case
                    when x is null then null
                    when trim(lower(x)) in ('', 'null', 'none', 'n/a', 'na', '-') then null
                    when trim(lower(x)) in ('py', 'python3') then 'python'
                    when trim(lower(x)) in ('pyspark', 'spark sql', 'apache spark') then 'spark'
                    when trim(lower(x)) in ('sql server', 'tsql', 't-sql') then 'sql'
                    when trim(lower(x)) in ('azure databricks', 'databricks workspace') then 'databricks'
                    when trim(lower(x)) in ('adls', 'adls gen2', 'azure data lake') then 'adls'
                    when trim(lower(x)) in ('terraform cloud', 'tf') then 'terraform'
                    when trim(lower(x)) in ('aws s3') then 's3'
                    else trim(lower(x))
                end
            ),
            x -> x is not null
        )
    """)
)

df = df.withColumn("skills_array", F.array_distinct(F.col("skills_array")))
df = df.withColumn("skills_array", F.array_sort(F.col("skills_array")))

#### Normalize Seniority and Job Family

In [0]:
# 8) Seniority e job family
df = (
    df
    .withColumn(
        "seniority",
        F.when(F.col("job_title").rlike(r"\bsenior\b|\bsr\b"), "senior")
         .when(F.col("job_title").rlike(r"\bpleno\b|\bmid\b|\bmiddle\b"), "mid")
         .when(F.col("job_title").rlike(r"\bjunior\b|\bjr\b"), "junior")
         .when(F.col("job_title").rlike(r"\best[aá]gio\b|\bintern\b"), "intern")
         .otherwise("unknown")
    )
    .withColumn(
        "job_family",
        F.when(F.col("job_title").rlike(r"data engineer"), "data engineering")
         .when(F.col("job_title").rlike(r"data analyst|bi analyst|analytics"), "data analytics")
         .when(F.col("job_title").rlike(r"data scientist"), "data science")
         .when(F.col("job_title").rlike(r"machine learning|ml engineer"), "machine learning")
         .when(F.col("job_title").rlike(r"analytics engineer"), "analytics engineering")
         .otherwise("other")
    )
)

#### Removing Duplicates

In [0]:
# 9) Remover duplicidades exatas
dedup_cols = [c for c in ["title", "company", "location_normalized", "salary_min", "salary_max"] if c in df.columns]

df = df.withColumn(
    "job_id",
    F.sha2(
        F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in dedup_cols]),
        256
    )
)

df = df.dropDuplicates(["job_id"])

#### Remove Similar Data

In [0]:
# 10) Tratar pequenos duplicados de vaga
window_small_dupes = Window.partitionBy("job_title", "company", "location").orderBy(F.col("ingestion_timestamp").desc())

df = (
    df
    .withColumn("rn_small_dupe", F.row_number().over(window_small_dupes))
    .filter(F.col("rn_small_dupe") == 1)
    .drop("rn_small_dupe")
)

#### Quality Flags

In [0]:
# 11) Criar flags de qualidade
df = (
    df
    .withColumn("is_title_null", F.col("job_title").isNull())
    .withColumn("is_company_null", F.col("company").isNull())
    .withColumn("is_location_null", F.col("location").isNull())
    .withColumn("is_salary_null", F.col("salary").isNull())
    .withColumn("has_skills", F.size(F.col("skills_array")) > 0)
)

df = df.filter(
    F.col("job_title").isNotNull() & F.col("company").isNotNull()
)

#### Create Silver Layer

In [0]:
# Silver is Ready
df_silver = df

# Save Silver as Parquet
silver_jobs_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/silver/jobs/"

(
    df_silver
    .write
    .mode("overwrite")
    .parquet(silver_jobs_path)
)

#### Generate Skills Table

In [0]:
# Criar a tabela de Skills
df_job_skills = (
    df_silver
    .select("job_id", F.explode_outer("skills_array").alias("skill"))
    .filter(F.col("skill").isNotNull())
)

# Save
silver_job_skills_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/silver/job_skills/"
(
    df_job_skills
    .write
    .mode("overwrite")
    .parquet(silver_job_skills_path)
)

#### Generate Quality Table

In [0]:
# Quality Table
df_quality = (
    df_silver
    .select(
        "job_id",
        "source_file",
        "ingestion_timestamp",
        "is_title_null",
        "is_company_null",
        "is_location_null",
        "is_salary_null",
        "has_skills"
    )
)

# Save Quality Table
silver_quality_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/silver/data_quality_jobs/"
(
    df_quality
    .write
    .mode("overwrite")
    .parquet(silver_quality_path)
)

### YES! It's Over!